In [1]:
import pandas as pd
import numpy as np
np.random.seed(42)
import random
random.seed(42)

from pdb import set_trace

from statistics import median, mean

from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from gensim.parsing.preprocessing import lower_to_unicode, preprocess_string, strip_tags, strip_punctuation, strip_multiple_whitespaces, strip_numeric

from tqdm.auto import tqdm
tqdm.pandas()

import re
from sklearn.metrics import silhouette_score

In [2]:
corpus = pd.read_pickle("../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_name.pkl.gz")
corpus.head()

,shop_id,ean,mpnr,price,product_id,name,cat_id,id,brand,shop_cat,...,desc,pzn,asin,title_temp,description_temp,title+desc,is_de,brand_temp,name+description+brand,has_long_name
0,10992,5.420070e+12,BONHS8007,1274.00,2820098210,"Vipack: Hochbett / Etagenbett ""BONNY"" Weiß / B...",1538,1817742672,Vipack,Möbel & Wohnen|Kindermöbel & Wohnen|Möbel|Bett...,...,"Vipack: Hochbett / Etagenbett ""BONNY"" Weiß / B...",NaN,NaN,"Vipack: Hochbett / Etagenbett ""BONNY"" Weiß / B...","Vipack: Hochbett / Etagenbett ""BONNY"" Weiß / B...","Vipack: Hochbett / Etagenbett ""BONNY"" Weiß / B...",True,Vipack,"vipack: hochbett / etagenbett ""bonny"" weiß / b...",True
1,10992,8.019227e+12,2761500,177.71,852480790,Sommerreifen PIRELLI P-ZERO (NEW) S.C. 215/45...,98581,4075272753,PRODUCT,Auto & Motorrad: Teile|Autoreifen & Felgen|Reifen,...,Sommerreifen PIRELLI P-ZERO (NEW) S.C. 215/45...,NaN,NaN,Sommerreifen PIRELLI P-ZERO (NEW) S.C. 215/45...,Sommerreifen PIRELLI P-ZERO (NEW) S.C. 215/45...,Sommerreifen PIRELLI P-ZERO (NEW) S.C. 215/45...,True,PRODUCT,sommerreifen pirelli p-zero (new) s.c. 215/45 ...,True
4,14657,4.897027e+12,NaN,63.41,1217832594,Telekom Speedphone 51 Festnetztelefon (mit Bas...,99625,2074416376,Deutsche Telekom,Elektronik & Computer > Smartphones & Telefoni...,...,Telekom Speedphone 51 Festnetztelefon (mit Bas...,NaN,NaN,Telekom Speedphone 51 Festnetztelefon (mit Bas...,Telekom Speedphone 51 Festnetztelefon (mit Bas...,Telekom Speedphone 51 Festnetztelefon (mit Bas...,True,Deutsche Telekom,telekom speedphone 51 festnetztelefon (mit bas...,True
6,5211,4.027181e+12,134105,85.57,1038150077,"Spiegelschrank »Basic« 60 cm weiß, Möbelpartne...",98626,1527509196,Möbelpartner,Büromöbel & Einrichten > Möbelelemente > Möbel...,...,"Spiegelschrank »Basic« 60 cm, Höhenausgleichss...",NaN,NaN,"Spiegelschrank »Basic« 60 cm weiß, Möbelpartne...","Spiegelschrank »Basic« 60 cm, Höhenausgleichss...","Spiegelschrank »Basic« 60 cm weiß, Möbelpartne...",True,Möbelpartner,"spiegelschrank »basic« 60 cm weiß, möbelpartne...",True
8,24880,4.014922e+12,NaN,999.99,3621538713,Gelenkarmmarkise SPETTMANN STAR Markisen Gr. 3...,142122,156275941980,BAUR,Markisen,...,"Produktdetails: UV-Schutzfaktor: UPF 80, Windw...",NaN,NaN,Gelenkarmmarkise SPETTMANN STAR Markisen Gr. 3...,"Produktdetails: UV-Schutzfaktor: UPF 80, Windw...",Gelenkarmmarkise SPETTMANN STAR Markisen Gr. 3...,True,BAUR,gelenkarmmarkise spettmann star markisen gr. 3...,True


In [3]:
counts = corpus['product_id'].value_counts()
counts = counts[counts > 3]

# Test to find the Correct Epsilon as this is dependent on the data

In [9]:
eps_list = [0.35] # To find the best eps
min_samples_list = [1]
results = [] # To help find best eps

for eps in eps_list:
    for min_sample in min_samples_list:
        print(f'eps: {eps}, min_samples: {min_sample}')
        
        corpus_selection = corpus[corpus['product_id'].isin(counts.index)].copy()
        print(corpus_selection.columns)
        corpus_selection = corpus_selection.drop(columns=["cat_id", "shop_cat"])
        corpus_selection = corpus_selection.drop_duplicates('product_id')
        CUSTOM_FILTERS = [lambda x: x.lower(), strip_tags, strip_punctuation, strip_multiple_whitespaces]

        corpus_selection['name_processed'] = corpus_selection['name'].apply(lower_to_unicode)
        corpus_selection['name_processed'] = corpus_selection['name_processed'].apply(preprocess_string, args=(CUSTOM_FILTERS,))
        corpus_selection['name_processed'] = corpus_selection['name_processed'].apply(lambda x: ' '.join(x))
        # TODO Why not drop duplicates directly here?????
        
        vectorizer = CountVectorizer(strip_accents='unicode', binary=True, min_df=4)
        #vectorizer = TfidfVectorizer(strip_accents='unicode', use_idf=False)
        matrix = vectorizer.fit_transform(corpus_selection['name_processed'])

        dbscan = DBSCAN(metric='cosine', eps=eps, min_samples=min_sample)
        #dbscan = OPTICS(metric='cosine', max_eps=eps, eps=eps, min_samples=min_sample, cluster_method='dbscan')
        clustering = dbscan.fit(matrix)
        corpus_selection['dbscan_cluster'] = clustering.labels_
        corpus_selection = corpus_selection.merge(
        corpus[['product_id', 'cat_id', 'shop_cat']],
        on='product_id',
        how='left'
        )
        
        counts_relevant = corpus['product_id'].value_counts()

        counts_relevant_unseen = counts_relevant[counts_relevant > 3]
        counts_relevant_unseen = counts_relevant_unseen[counts_relevant_unseen < 7]
        
        counts_relevant_seen = counts_relevant[counts_relevant > 6]
        counts_relevant_seen = counts_relevant_seen[counts_relevant_seen < 81]
        
        print(f'Seen data:')
        corpus_selection_seen = corpus_selection[corpus_selection['product_id'].isin(counts_relevant_seen.index)].copy()
        corpus_selection_seen = corpus_selection_seen[corpus_selection_seen['dbscan_cluster'] != -1]
        
        print(f'Clusters found: {len(corpus_selection_seen["dbscan_cluster"].unique())}')
        print(f'Mean cluster size: {mean(corpus_selection_seen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_seen["dbscan_cluster"].value_counts())}')
        
        counts_clustering = corpus_selection_seen['dbscan_cluster'].value_counts()
        counts_clustering = counts_clustering[counts_clustering > 2]
        corpus_selection_seen = corpus_selection_seen[corpus_selection_seen['dbscan_cluster'].isin(counts_clustering.index)]
        corpus_selection_seen = corpus_selection_seen.sort_values('dbscan_cluster')
        
        print(f'Clusters >2 found: {len(corpus_selection_seen["dbscan_cluster"].unique())}')
        print(f'Mean cluster size: {mean(corpus_selection_seen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_seen["dbscan_cluster"].value_counts())}\n')
        corpus_selection_seen = corpus_selection_seen[['dbscan_cluster', 'brand', 'name', 'desc', 'price', 'dlv_time' ,'shop_id', 'ean', 'mpnr',
       'cat_id', 'id', 'product_id', 'shop_cat', 'aid', 'pzn', 'asin']]
        
        # Define regex for illegal control chars (openpyxl disallows ASCII < 32 except tab, newline, carriage return)
        ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]')

        # Apply cleaning to all string cells
        corpus_selection_seen = corpus_selection_seen.map(
            lambda x: ILLEGAL_CHARACTERS_RE.sub("", x) if isinstance(x, str) else x
        )

        corpus_selection_seen.to_csv(f'../data/working/dbscan/seen_dbscan_eps{eps}_minsamples{min_sample}_dedup_preprocessed_rev2_docs_since_2020_01_01_only_en_strict_only_long_title_only_mainentity.csv',
        index=False)

        db_clu = corpus_selection_seen[['product_id', 'dbscan_cluster']].copy()
        db_clu = db_clu.drop_duplicates('product_id')
        db_clu.to_csv(f'../data/working/dbscan/seen_dbscan_mapping.csv', header=True, index=False)
        db_clu = corpus_selection_seen['dbscan_cluster'].copy()
        db_clu = db_clu.drop_duplicates()
        db_clu = db_clu.sort_values()
        db_clu.to_csv(f'../data/working/dbscan/seen_dbscan_clusters.csv', header=True, index=False)
        

        print(f'Unseen data:')
        corpus_selection_unseen = corpus_selection[corpus_selection['product_id'].isin(counts_relevant_unseen.index)].copy()
        corpus_selection_unseen = corpus_selection_unseen[corpus_selection_unseen['dbscan_cluster'] != -1]
        
        print(f'Clusters found: {len(corpus_selection_unseen["dbscan_cluster"].unique())}')
        print(f'Mean cluster size: {mean(corpus_selection_unseen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_unseen["dbscan_cluster"].value_counts())}')
        
        counts_clustering = corpus_selection_unseen['dbscan_cluster'].value_counts()
        counts_clustering = counts_clustering[counts_clustering > 2]
        corpus_selection_unseen = corpus_selection_unseen[corpus_selection_unseen['dbscan_cluster'].isin(counts_clustering.index)]
        corpus_selection_unseen = corpus_selection_unseen.sort_values('dbscan_cluster')
        
        print(f'Clusters >2 found: {len(corpus_selection_unseen["dbscan_cluster"].unique())}')
        print(f'Mean cluster size: {mean(corpus_selection_unseen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_unseen["dbscan_cluster"].value_counts())}\n')
        corpus_selection_unseen = corpus_selection_unseen[['dbscan_cluster', 'brand', 'name', 'desc', 'price', 'dlv_time' ,'shop_id', 'ean', 'mpnr',
       'cat_id', 'id', 'product_id', 'shop_cat', 'aid', 'pzn', 'asin']]
        
        # Clean before writing unseen
        corpus_selection_unseen = corpus_selection_unseen.map(
            lambda x: ILLEGAL_CHARACTERS_RE.sub("", x) if isinstance(x, str) else x
        )

        corpus_selection_unseen.to_csv(f'../data/working/dbscan/unseen_dbscan_eps{eps}_minsamples{min_sample}_dedup_preprocessed_rev2_docs_since_2020_01_01_only_en_strict_only_long_title_only_mainentity.csv', index=False)
        
        db_clu = corpus_selection_unseen[['product_id', 'dbscan_cluster']].copy()
        db_clu = db_clu.drop_duplicates('product_id')
        db_clu.to_csv(f'../data/working/dbscan/unseen_dbscan_mapping.csv', header=True, index=False)
        db_clu = corpus_selection_unseen['dbscan_cluster'].copy()
        db_clu = db_clu.drop_duplicates()
        db_clu = db_clu.sort_values()
        db_clu.to_csv(f'../data/working/dbscan/unseen_dbscan_clusters.csv', header=True, index=False)

        n_clusters_seen = len(corpus_selection_seen["dbscan_cluster"].unique())
        mean_size_seen = mean(corpus_selection_seen["dbscan_cluster"].value_counts()) if n_clusters_seen > 0 else 0
        median_size_seen = median(corpus_selection_seen["dbscan_cluster"].value_counts()) if n_clusters_seen > 0 else 0
        noise_ratio_seen = np.sum(clustering.labels_ == -1) / len(clustering.labels_)

        n_clusters_unseen = len(corpus_selection_unseen["dbscan_cluster"].unique())
        mean_size_unseen = mean(corpus_selection_unseen["dbscan_cluster"].value_counts()) if n_clusters_unseen > 0 else 0
        median_size_unseen = median(corpus_selection_unseen["dbscan_cluster"].value_counts()) if n_clusters_unseen > 0 else 0
        if n_clusters_seen > 1:
            sil = silhouette_score(matrix, clustering.labels_, metric="cosine")
        else:
            sil = np.nan
            
        results.append({
            "eps": eps,
            "min_samples": min_sample,
            "#clusters_seen": n_clusters_seen,
            "mean_size_seen": mean_size_seen,
            "median_size_seen": median_size_seen,
            "#clusters_unseen": n_clusters_unseen,
            "mean_size_unseen": mean_size_unseen,
            "median_size_unseen": median_size_unseen,
            "noise_ratio": round(noise_ratio_seen, 3),
            "silhouette": round(sil, 3) if not np.isnan(sil) else None
        })

        print(f'-------------------------------------------------------------------------')

eps: 0.35, min_samples: 1


Index(['shop_id', 'ean', 'mpnr', 'price', 'product_id', 'name', 'cat_id', 'id',
       'brand', 'shop_cat', 'aid', 'dlv_time', 'desc', 'pzn', 'asin',
       'title_temp', 'description_temp', 'title+desc', 'is_de', 'brand_temp',
       'name+description+brand', 'has_long_name'],
      dtype='object')
Seen data:
Clusters found: 45814
Mean cluster size: 32.53800148426245, Median cluster_size: 12.0
Clusters >2 found: 45814
Mean cluster size: 32.53800148426245, Median cluster_size: 12.0

Unseen data:
Clusters found: 51646
Mean cluster size: 12.236649498509081, Median cluster_size: 5.0
Clusters >2 found: 51646
Mean cluster size: 12.236649498509081, Median cluster_size: 5.0

-------------------------------------------------------------------------


In [10]:
print(results)


[{'eps': 0.35, 'min_samples': 1, '#clusters_seen': 45814, 'mean_size_seen': 32.53800148426245, 'median_size_seen': 12.0, '#clusters_unseen': 51646, 'mean_size_unseen': 12.236649498509081, 'median_size_unseen': 5.0, 'noise_ratio': 0.0, 'silhouette': -0.124}]
